In [1]:

import re
import nltk
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/dinesh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

In [4]:
def tfidf_filter(resume, job_list, top_k=15):
    documents = [resume] + job_list
    
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(documents)
    
    resume_vec = tfidf_matrix[0]
    
    job_vecs = tfidf_matrix[1:]
    
    similarities = cosine_similarity(resume_vec, job_vecs)[0]
    
    
    top_indices = similarities.argsort()[::-1][:top_k]
    
    return top_indices, similarities

In [5]:
model = SentenceTransformer('all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
def bert_rerank(resume, job_list, top_indices):
    resume_emb = model.encode(resume)
    
    results = []
    
    for idx in top_indices:
        job_emb = model.encode(job_list[idx])
        
        sim = cosine_similarity([resume_emb], [job_emb])[0][0]
        score = sim * 100
        
        results.append((idx, score))
    
    
    results.sort(key=lambda x: x[1], reverse=True)
    
    return results
    

In [48]:
def job_matching_pipeline(resume_text, job_descriptions, top_k=15):
    
   
    
    resume_clean = preprocess(resume_text)
    jobs_clean = [preprocess(j) for j in job_descriptions]
    
    
    top_indices, tfidf_scores = tfidf_filter(resume_clean, jobs_clean, top_k)
    
    
    bert_results = bert_rerank(resume_clean, jobs_clean, top_indices)
    
    
    final_results = []
    
    for idx, score in bert_results:
        
        final_results.append({
            "job_id": idx,
            "match_score": round(score, 2),
            
            
        })
    
    return final_results

In [47]:
jobs = [
    "Looking for Python developer with ML and SQL",
    "Frontend developer with React skills",
    "Data scientist with machine learning and Python",
    "Java backend developer",
    "AI engineer with deep learning experience"
]

result = job_matching_pipeline(user['resumeText'], jobs)

for r in result[:3]:
    print(r)

{'job_id': np.int64(1), 'match_score': np.float32(42.71), 'job_description': 'Frontend developer with React skills'}
{'job_id': np.int64(3), 'match_score': np.float32(38.63), 'job_description': 'Java backend developer'}
{'job_id': np.int64(0), 'match_score': np.float32(20.95), 'job_description': 'Looking for Python developer with ML and SQL'}


In [25]:
from pymongo import MongoClient

uri = ""

client = MongoClient(uri)

db = client["hirewire"]
collection =db["candidates"]

print("Connected to Atlas!")

Connected to Atlas!


In [26]:
data = collection.find()

print(data)

In [29]:
for doc in data:
    print(doc.resumeText)
    

In [37]:
collectionJobs =db["jobs"]

In [40]:
allJobdetails=list(collectionJobs.find())

In [43]:
jD=[]
for jobDesc in allJobdetails:
    jD.append(jobDesc['rawDescription'])

In [50]:
result = job_matching_pipeline(user['resumeText'], jD)

for r in result[:10]:
    print(r)

{'job_id': np.int64(7), 'match_score': np.float32(72.93)}
{'job_id': np.int64(6), 'match_score': np.float32(71.17)}
{'job_id': np.int64(9), 'match_score': np.float32(66.75)}
{'job_id': np.int64(0), 'match_score': np.float32(66.13)}
{'job_id': np.int64(1), 'match_score': np.float32(57.6)}
{'job_id': np.int64(8), 'match_score': np.float32(54.56)}
{'job_id': np.int64(2), 'match_score': np.float32(54.37)}
{'job_id': np.int64(4), 'match_score': np.float32(53.79)}
{'job_id': np.int64(5), 'match_score': np.float32(50.64)}
{'job_id': np.int64(3), 'match_score': np.float32(49.88)}
